In [0]:
%sql

CREATE CATALOG IF NOT EXISTS medalhao_credit;

In [0]:
%sql

USE CATALOG medalhao_credit;

CREATE SCHEMA IF NOT EXISTS silver_credit;

In [0]:
%sql
USE SCHEMA silver_credit;

In [0]:
from pyspark.sql import SparkSession

from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, TimestampType
from pyspark.sql.functions import col, trim, regexp_replace, initcap, when, lower, lit

import requests
import pandas as pd
from datetime import datetime
import time

In [0]:
catalogo = "medalhao_credit"
bronze_db_name = "bronze_credit"
silver_db_name = "silver_credit"

volume_path = "/Volumes/workspace/default/data"

print(f"Ambiente configurado: {catalogo}.{silver_db_name}")

In [0]:
# Leitura da tabela Bronze de Chamados
df_bronze = spark.table(f"{catalogo}.{bronze_db_name}.chamados")

display(df_bronze.limit(5))

In [0]:
# Célula de Diagnóstico
print("Lista exata de colunas:")
print(df_bronze.columns)

In [0]:
df_ordenado = (
    df_bronze
    .withColumnRenamed("_c0", "id_chamado") #Renomeia a coluna
    .withColumn("id_chamado", col("id_chamado").cast("int")) #Transforma tudo em int
    .filter(col("id_chamado").isNotNull())  #Remove linhas se o ID estiver vazio (lixo)
    .dropDuplicates(["id_chamado"])         #Se tiver dois IDs iguais, mantém apenas um
    .orderBy("id_chamado")
)

display(df_ordenado)

In [0]:
df_cliente_tratado = (
    df_ordenado # Continuando do passo anterior
    
    # 1. Renomear
    .withColumnRenamed("_c1", "id_cliente")
    
    # 2. Tipagem SEGURA (Long em vez de Int para não quebrar CPFs)
    .withColumn("id_cliente", col("id_cliente").cast("long"))
    
    # 3. Tratamento de Nulos (Regra de Ouro)
    # Não apaga a linha (o chamado existiu), mas marca o cliente como -1 (Desconhecido)
    .fillna(-1, subset=["id_cliente"])
)

display(df_cliente_tratado)

In [0]:
df_motivo_tratado = (
    df_cliente_tratado # Continua do passo de ID_Cliente
    
    # 1. Renomear
    .withColumnRenamed("_c2", "motivo")
    
    # 2. Tipagem e Trim (Limpeza básica)
    .withColumn("motivo", trim(col("motivo").cast("string")))
    
    # 3. CIRURGIA DE RECONSTRUÇÃO (Regex)
    # O ponto (.) substitui o caractere estragado. 
    
    .withColumn("motivo", regexp_replace(col("motivo"), "Contrata..o", "Contratacao"))
    .withColumn("motivo", regexp_replace(col("motivo"), "Contesta..o", "Contestacao"))
    .withColumn("motivo", regexp_replace(col("motivo"), "Altera..o", "Alteracao"))
    .withColumn("motivo", regexp_replace(col("motivo"), "cart.o", "cartao"))
    .withColumn("motivo", regexp_replace(col("motivo"), "D.vidas", "Duvidas"))
    .withColumn("motivo", regexp_replace(col("motivo"), "Informa..es", "Informacoes"))
    .withColumn("motivo", regexp_replace(col("motivo"), "Solicita..o", "Solicitacao"))

    # \\b significa "borda da palavra". Só pega se começar e terminar ali.
    .withColumn("motivo", regexp_replace(col("motivo"), "(?i)\\bn.o\\b", "Nao"))
    
    # 4. Padronização Visual (Capitalize)
    # Deixa "duvidas gerais" -> "Duvidas Gerais"
    .withColumn("motivo", initcap(col("motivo")))
    
    # 5. Tratamento de Nulos
    # Regra: Motivo vazio vira "Motivo Nao Informado"
    .fillna("Motivo Nao Informado", subset=["motivo"])
    .withColumn("motivo", 
                when((col("motivo") == "") | (col("motivo").isNull()), "Motivo Nao Informado")
                .otherwise(col("motivo")))
)

display(df_motivo_tratado)

In [0]:
df_canal_tratado = (
    df_motivo_tratado # Continua do passo anterior
    
    # 1. Renomear
    .withColumnRenamed("_c3", "canal")
    
    # 2. Tipagem e Trim
    .withColumn("canal", trim(col("canal").cast("string")))
    
    # 3. NORMALIZAÇÃO E PADRONIZAÇÃO
    .withColumn("canal", 
                
                # Regra 1: Chatbot
                when(lower(col("canal")).like("%chat%"), "Chatbot")
                
                # Regra 2: URA (Pega URA ou U.r.a)
                .when((lower(col("canal")).like("%ura%")) | (lower(col("canal")).like("%u.r.a%")), "URA")
                
                # Regra 3: Web e Email (Adicionei conforme sua lista)
                .when(lower(col("canal")).like("%web%"), "Web")
                .when(lower(col("canal")).like("%mail%"), "Email")
                
                # Regra 4: ATENDIMENTO ESPECIALIZADO (Checa ANTES do Inicial)
                # Se tiver a palavra "especializado" em qualquer lugar, classifica aqui
                .when(lower(col("canal")).like("%especializ%"), "Atendimento Especializado")
                
                # Regra 5: ATENDIMENTO INICIAL
                # Pega "Inicial", "Atend. Inicial", ou qualquer "Atend" genérico que sobrou
                .when((lower(col("canal")).like("%inici%")) | (lower(col("canal")).like("%atend%")), "Atendimento Inicial")
                
                .otherwise(initcap(col("canal")))
               )

    # 4. Tratamento de Nulos
    .fillna("Canal Nao Identificado", subset=["canal"])
    .withColumn("canal", 
                when((col("canal") == "") | (col("canal").isNull()), "Canal Nao Identificado")
                .otherwise(col("canal")))
)

# Validação Final
print("Validação: Verifique se Especializado e Inicial estão separados:")
df_canal_tratado.groupBy("canal").count().show(truncate=False)

display(df_canal_tratado)

In [0]:
df_resolvido_tratado = (
    df_canal_tratado # Continua do passo anterior (Canal)
    
    # 1. Renomear (Snake Case)
    .withColumnRenamed("_c4", "resolvido")
    
    # 2. Tipagem e Trim
    .withColumn("resolvido", trim(col("resolvido").cast("string")))
    
    # 3. NORMALIZAÇÃO BINÁRIA (melhor prática)
    # Estratégia: Em vez de brigar com o acento, usamos a lógica do "Contém S"
    .withColumn("resolvido", 
                
                # Regra 1: Se tiver "s" ou "S" (Sim, S, yes), vira "Sim"
                when(lower(col("resolvido")).like("%s%"), "Sim")
                
                # Regra 2: Se tiver "n" ou "N" (Nao, No, No), vira "Nao"
                .when(lower(col("resolvido")).like("%n%"), "Nao")
                
                # Caso contrário (Vazio ou Lixo), vira "Nao Informado"
                .otherwise("Nao Informado")
               )

    # 4. Tratamento de Nulos (Garantia Extra)
    # Se sobrar algum null real, vira "Nao Informado"
    .fillna("Nao Informado", subset=["resolvido"])
)

# Validação: Deve aparecer APENAS "Sim", "Nao" e talvez "Nao Informado"
print("Distribuição da coluna Resolvido:")
df_resolvido_tratado.groupBy("resolvido").count().show()

display(df_resolvido_tratado)

In [0]:
df_hora_abertura_tratado = (
    df_resolvido_tratado # Continua do passo anterior
    
    # 1. Renomear (Snake Case e Descritivo)
    .withColumnRenamed("_c5", "hora_abertura_chamado")
    
    # 2. Tipagem Forte (Schema Enforcement)
    # Mesmo que esteja tudo Null ou vazio, o tipo é Timestamp.
    .withColumn("hora_abertura_chamado", col("hora_abertura_chamado").cast("timestamp"))
)

# Validação (quero ver o schema como "timestamp" e os dados como "null")
print("Schema da coluna:")
df_hora_abertura_tratado.select("hora_abertura_chamado").printSchema()

print("\nVisualização dos dados (Devem estar null):")
df_hora_abertura_tratado.select("hora_abertura_chamado").show(5)

display(df_hora_abertura_tratado)

In [0]:
df_inicio_tratado = (
    df_hora_abertura_tratado # Continua do passo anterior
    
    # 1. Renomear
    .withColumnRenamed("_c6", "hora_inicio_atendimento")
    
    # 2. Trim (Limpeza de espaços)
    .withColumn("hora_inicio_atendimento", trim(col("hora_inicio_atendimento")))
    
    # 3. Lógica de Negócio (A cópia condicional)
    # Se o texto disser "igual...", ele busca o valor da coluna hora_abertura_chamado.
    .withColumn("hora_inicio_atendimento", 
                when(lower(col("hora_inicio_atendimento")).like("%igual%"), col("hora_abertura_chamado"))
                .otherwise(col("hora_inicio_atendimento")))
    
    # 4. Tipagem Final (Schema Enforcement)
    # Tudo que não for data válida vira Null automaticamente aqui
    .withColumn("hora_inicio_atendimento", col("hora_inicio_atendimento").cast("timestamp"))
)

# Validação:
print("Schema atualizado:")
df_inicio_tratado.select("hora_inicio_atendimento").printSchema()

display(df_inicio_tratado)

In [0]:
df_fim_tratado = (
    df_inicio_tratado # Continua do passo anterior
    
    # 1. Renomear
    .withColumnRenamed("_c7", "hora_finalizacao_atendimento")
    
    # 2. Trim (Limpeza básica de espaços invisíveis)
    .withColumn("hora_finalizacao_atendimento", trim(col("hora_finalizacao_atendimento")))
    
    # 3. Tipagem (Schema Enforcement)
    # Transforma texto/vazio em Data Real.
    .withColumn("hora_finalizacao_atendimento", col("hora_finalizacao_atendimento").cast("timestamp"))
)

# Validação do Schema
print("Schema final das colunas de tempo:")
df_fim_tratado.select("hora_abertura_chamado", 
                      "hora_inicio_atendimento", 
                      "hora_finalizacao_atendimento").printSchema()

display(df_fim_tratado)